# Exportação da Gold para o portal — V-Energy

Lê as **12 tabelas Gold do contrato** (`contracts/tabelas/`), confere o lote inteiro com as mesmas regras do importador do portal e grava, num Volume, um `<tabela>.csv` por tabela e o `manifest.json`. É o pacote que o `importador` carrega no banco do portal no lugar da amostra `contracts/samples` (#96).

- **Leitura:** as 10 tabelas Delta numa versão fixa (`VERSION AS OF`); as duas da Gold de faturas pelo marcador `tb_gold_faturas_publicacao`, como pede `docs/gold/faturas.md`.
- **Conferência:** colunas, tipos e ordem do YAML; cada valor como o `parse` do importador; chave única; chaves estrangeiras; os portões do importador (R2, R3, R4, inspeções, datas e reconciliação da receita); período, `data_referencia` igual à da Gold de faturas e R6.
- **Exportação** (`modo=exportar`): uma pasta nova por lote, os 12 CSVs, `fontes.json` (versões lidas) e, por último, o `manifest.json` gerado por `contracts/manifesto.py`.

Parâmetros: `data_referencia` (AAAA-MM-DD, a mesma da Gold de faturas), `diretorio_exportacao` (pasta de um Volume), `batch_id` (opcional) e `modo` (`validar`, o padrão, não grava nada). Detalhes em `docs/gold/exportacao.md`.

In [0]:
"""Exportação das 12 tabelas Gold do contrato num lote que o importador do portal carrega."""

import csv
import importlib.util
import json
import re
from decimal import Decimal
from itertools import zip_longest
from pathlib import Path

import yaml
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

VERSAO_REGRAS = 'GOLD_EXPORTACAO_V1'
VERSAO_CONTRATO = '0.3.0-provisional'
# Publicadas pela Gold de faturas em revisões: o marcador diz qual vale (docs/gold/faturas.md)
TABELAS_FATURAS = ('uc_consumo_faturamento_mensal', 'receita_volume_mensal')
CONFIRMADA = 'IRREGULARIDADE CONFIRMADA'
TIPOS_SPARK = {
    'text': 'string',
    'date': 'date',
    'timestamp': 'timestamp_ntz',
    'boolean': 'boolean',
    'integer': 'int',
    'jsonb': 'array<string>',
}


def tipo_spark(tipo: str) -> str:
    decimal = re.fullmatch(r'numeric\((\d+),\s*(\d+)\)', tipo)
    if decimal:
        return f'decimal({decimal[1]},{decimal[2]})'
    if tipo not in TIPOS_SPARK:
        raise ValueError(f'Tipo {tipo} do contrato sem equivalente no Spark')
    return TIPOS_SPARK[tipo]


def ler_contratos(raiz: Path) -> dict:
    """As 12 especificações de contracts/tabelas, na versão para a qual a exportação foi escrita."""
    versao = (raiz / 'VERSION').read_text().strip()
    if versao != VERSAO_CONTRATO:
        raise ValueError(f'Exportação escrita para o contrato {VERSAO_CONTRATO}, não {versao}')
    specs = {}
    for caminho in sorted((raiz / 'tabelas').glob('*.yaml')):
        spec = yaml.safe_load(caminho.read_text(encoding='utf-8'))
        if spec.get('table') != caminho.stem:
            raise ValueError(f'{caminho.name}: table diferente do nome do arquivo')
        for nome, coluna in spec['columns'].items():
            if not {'type', 'nullable', 'description'} <= set(coluna):
                raise ValueError(f'{caminho.name}: coluna {nome} sem type, nullable ou description')
            tipo_spark(coluna['type'])
        specs[caminho.stem] = spec
    if len(specs) != 12:
        raise ValueError(f'O contrato tem {len(specs)} tabelas; a exportação espera 12')
    return specs


def carregar_gerador_de_manifesto(raiz: Path):
    """O gerador do próprio contrato (contracts/manifesto.py), sem copiar a regra."""
    especificacao = importlib.util.spec_from_file_location('manifesto', raiz / 'manifesto.py')
    modulo = importlib.util.module_from_spec(especificacao)
    especificacao.loader.exec_module(modulo)
    return modulo.gerar_manifesto


def contar(df: DataFrame, regras: dict) -> dict:
    """Conta, numa passada, as linhas que violam cada regra (verdadeira = violação)."""
    if not regras:
        return {}
    linha = df.agg(*[F.count(F.when(cond, 1)).alias(nome) for nome, cond in regras.items()])
    return linha.first().asDict()


def divergencias_de_esquema(df: DataFrame, spec: dict) -> list:
    """Colunas, tipos e ordem do YAML; o CSV sai nessa ordem e o importador confere o cabeçalho."""
    esperado = [(c, tipo_spark(v['type'])) for c, v in spec['columns'].items()]
    gerado = [(c.name, c.dataType.simpleString()) for c in df.schema]
    return [f'gerado {g}, esperado {e}' for g, e in zip_longest(gerado, esperado) if g != e]


def regras_da_tabela(nome: str, spec: dict) -> dict:
    """O que o parse do importador cobra de cada valor (importador/src/importador/contracts.py)."""
    regras = {}
    for coluna, c in spec['columns'].items():
        valor, rotulo = F.col(coluna), f'{nome}.{coluna}'
        if not c['nullable']:
            regras[f'{rotulo}: nulo'] = valor.isNull()
        if c['type'] == 'text':
            regras[f'{rotulo}: texto vazio'] = F.trim(valor) == ''
        if c['type'] == 'jsonb':
            regras[f'{rotulo}: lista vazia ou com texto vazio'] = (F.size(valor) == 0) | F.exists(
                valor, lambda x: x.isNull() | (F.trim(x) == '')
            )
        if c['type'] == 'timestamp':
            # O CSV leva a data-hora até os segundos
            regras[f'{rotulo}: fração de segundo'] = F.date_format(valor, 'SSSSSS') != '000000'
        if c.get('month'):
            regras[f'{rotulo}: competência fora do dia 1'] = F.dayofmonth(valor) != 1
        if c.get('pattern'):
            # O grupo entre ^ e \z casa o valor inteiro, como o re.fullmatch do importador
            regras[f'{rotulo}: fora do padrão {c["pattern"]}'] = ~valor.rlike(
                rf'^(?:{c["pattern"]})\z'
            )
        if 'enum' in c:
            regras[f'{rotulo}: fora do domínio'] = ~valor.isin(c['enum'])
        if 'minimum' in c:
            regras[f'{rotulo}: abaixo de {c["minimum"]}'] = valor < c['minimum']
        if 'maximum' in c:
            regras[f'{rotulo}: acima de {c["maximum"]}'] = valor > c['maximum']
    return regras


def chave_repetida(df: DataFrame, spec: dict) -> int:
    return df.count() - df.select(*spec['primary_key']).distinct().count()


def referencias_orfas(tabelas: dict, specs: dict) -> dict:
    """As chaves estrangeiras (references) que o importador cria no banco."""
    orfas = {}
    for nome, spec in specs.items():
        for coluna, c in spec['columns'].items():
            if 'references' not in c:
                continue
            alvo, chave = c['references']
            origem = (
                tabelas[nome].filter(F.col(coluna).isNotNull()).select(F.col(coluna).alias('k'))
            )
            orfas[f'{nome}.{coluna}: sem {alvo}.{chave} correspondente'] = origem.join(
                tabelas[alvo].select(F.col(chave).alias('k')), 'k', 'left_anti'
            ).count()
    return orfas


# Portões do importador (importador/src/importador/loader.py, validate) escritos em Spark SQL
# sobre as views exp_<tabela>; cada consulta devolve as linhas que o importador reprovaria.
PORTOES = {
    'inspecoes_historicas: resultado, tipo e kWh incoerentes': f"""
        SELECT 1 FROM exp_inspecoes_historicas WHERE NOT (
          (resultado = '{CONFIRMADA}' AND tipo_irregularidade IS NOT NULL
            AND kwh_recuperado > 0) OR
          (resultado = 'SEM IRREGULARIDADE' AND tipo_irregularidade IS NULL
            AND kwh_recuperado = 0))""",
    'medidores_historico: retirada antes da instalação': """
        SELECT 1 FROM exp_medidores_historico WHERE data_retirada < data_instalacao""",
    'ocorrencias_uc: datas ou durações incoerentes': """
        SELECT 1 FROM exp_ocorrencias_uc
        WHERE data_encerramento < data_abertura OR duracao_efetiva_min > duracao_min
          OR round(CAST(timestampdiff(SECOND, data_abertura, data_encerramento)
                        AS DECIMAL(20, 0)) / 60, 2) <> duracao_min""",
    'uc_consumo_faturamento_mensal: competência sem fatura incoerente (R2)': """
        SELECT 1 FROM exp_uc_consumo_faturamento_mensal
        WHERE (id_fatura IS NULL AND NOT (
          bandeira IS NULL AND status_pagamento IS NULL AND data_vencimento IS NULL
          AND tarifa_base IS NULL AND adicional_bandeira IS NULL
          AND volume_faturado_kwh = 0 AND valor_total = 0 AND valor_arrecadado = 0
        )) OR (id_fatura IS NOT NULL AND (
          bandeira IS NULL OR status_pagamento IS NULL OR data_vencimento IS NULL
          OR tarifa_base IS NULL OR adicional_bandeira IS NULL))""",
    'uc_consumo_faturamento_mensal: arrecadado incompatível com o status (R2)': """
        SELECT 1 FROM exp_uc_consumo_faturamento_mensal
        WHERE (status_pagamento = 'PAGA' AND valor_arrecadado <> valor_total)
           OR (status_pagamento = 'ABERTA' AND valor_arrecadado <> 0)""",
    'uc_consumo_faturamento_mensal: troca de medidor com leitura utilizável (R2)': """
        SELECT 1 FROM exp_uc_consumo_faturamento_mensal
        WHERE flag_troca_medidor = true AND metodo_consumo = 'LEITURA_UTILIZAVEL'""",
    'uc_consumo_faturamento_mensal: leitura utilizável com consumo alterado (R2)': """
        SELECT 1 FROM exp_uc_consumo_faturamento_mensal
        WHERE metodo_consumo = 'LEITURA_UTILIZAVEL'
          AND consumo_kwh_analitico <> consumo_kwh_original""",
    'uc_consumo_faturamento_mensal: volume faturado diferente do consumo (R2)': """
        SELECT 1 FROM exp_uc_consumo_faturamento_mensal
        WHERE id_fatura IS NOT NULL AND volume_faturado_kwh <> consumo_kwh_analitico""",
    'uc_consumo_faturamento_mensal: grupo_uc fora da regra temporal (R3)': f"""
        WITH primeira_irregularidade AS (
          SELECT id_uc, trunc(min(data), 'MM') AS mes_irregularidade
          FROM exp_inspecoes_historicas WHERE resultado = '{CONFIRMADA}' GROUP BY id_uc
        ), esperado AS (
          SELECT CASE
              WHEN g.id_uc IS NOT NULL AND m.competencia >= trunc(g.data_conexao, 'MM')
                THEN 'GD'
              WHEN i.id_uc IS NOT NULL AND m.competencia >= i.mes_irregularidade
                THEN 'IRREGULARIDADE_CONFIRMADA'
              ELSE 'DEMAIS'
            END AS grupo_esperado, m.grupo_uc
          FROM exp_uc_consumo_faturamento_mensal m
          LEFT JOIN exp_gd_uc g ON m.id_uc = g.id_uc
          LEFT JOIN primeira_irregularidade i ON m.id_uc = i.id_uc
        )
        SELECT 1 FROM esperado WHERE grupo_esperado <> grupo_uc""",
    'dim_uc: flag_gd diferente da presença em gd_uc (R4)': """
        SELECT 1 FROM exp_dim_uc u FULL OUTER JOIN exp_gd_uc g ON u.id_uc = g.id_uc
        WHERE (u.flag_gd = true AND g.id_uc IS NULL)
           OR (u.flag_gd = false AND g.id_uc IS NOT NULL) OR u.id_uc IS NULL""",
    'fila_inspecao: UC com GD (R4)': """
        SELECT 1 FROM exp_fila_inspecao f JOIN exp_gd_uc g ON f.id_uc = g.id_uc""",
    'continuidade_alimentador_mensal: numero_ucs diferente da dim_alimentador (R4)': """
        SELECT 1 FROM exp_continuidade_alimentador_mensal c
        JOIN exp_dim_alimentador a USING (alimentador) WHERE c.numero_ucs <> a.numero_ucs""",
    'receita_volume_mensal: não reconcilia por grão com uc_consumo e dim_uc': """
        WITH detalhe AS (
          SELECT m.competencia, u.municipio, u.bairro, u.classe, m.grupo_uc,
            sum(m.volume_faturado_kwh) volume_faturado_kwh,
            sum(m.valor_total) receita, sum(m.valor_arrecadado) receita_arrecadada,
            round(sum(m.volume_faturado_kwh * coalesce(m.tarifa_base, 0)), 2)
              componente_tarifa_reais,
            round(sum(m.volume_faturado_kwh * coalesce(m.adicional_bandeira, 0)), 2)
              componente_bandeira_reais
          FROM exp_uc_consumo_faturamento_mensal m JOIN exp_dim_uc u USING (id_uc)
          GROUP BY 1, 2, 3, 4, 5
        ), agregado AS (SELECT * FROM exp_receita_volume_mensal)
        (SELECT * FROM detalhe EXCEPT SELECT * FROM agregado)
        UNION ALL (SELECT * FROM agregado EXCEPT SELECT * FROM detalhe)""",
}


def como_texto(coluna: str, tipo: str):
    """O valor no formato que o parse do importador lê; nulo vira campo vazio no CSV."""
    valor = F.col(coluna)
    if tipo == 'date':
        return F.date_format(valor, 'yyyy-MM-dd')
    if tipo == 'timestamp':
        return F.date_format(valor, "yyyy-MM-dd'T'HH:mm:ss")
    if tipo == 'boolean':
        return F.when(valor, 'true').when(~valor, 'false')
    if tipo == 'jsonb':
        return F.to_json(valor)
    return valor.cast('string')


def escrever_csv(df: DataFrame, spec: dict, arquivo: Path) -> int:
    """CSV com o csv do Python, o mesmo módulo que o importador usa para ler, em ordem de chave."""
    colunas = list(spec['columns'])
    texto = df.select(*[como_texto(c, spec['columns'][c]['type']).alias(c) for c in colunas])
    linhas = 0
    with arquivo.open('x', newline='', encoding='utf-8') as destino:
        escritor = csv.writer(destino, lineterminator='\n')
        escritor.writerow(colunas)
        for linha in texto.orderBy(*spec['primary_key']).toLocalIterator():
            escritor.writerow(['' if v is None else v for v in linha])
            linhas += 1
    return linhas


def totais_de_controle(df: DataFrame, spec: dict) -> dict:
    colunas = spec.get('control_totals', [])
    if not colunas:
        return {}
    linha = df.agg(*[F.coalesce(F.sum(c), F.lit(0)).alias(c) for c in colunas]).first()
    return {c: Decimal(str(linha[c])) for c in colunas}

In [0]:
# ruff: noqa: F821
import uuid
from datetime import date

parametros = {
    'schema_gold': 'v-energy.gold',
    # Relativo à pasta do notebook no repositório (ou no bundle de data-engineering/)
    'contratos': '../../contracts',
    # Retrato de status da extração, o mesmo da Gold de faturas (docs/gold/faturas.md)
    'data_referencia': '',
    # Pasta de um Volume; cada lote vai para uma subpasta nova, com o batch_id
    'diretorio_exportacao': '',
    'batch_id': '',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'exportar'}:
    raise ValueError('modo deve ser validar ou exportar')
try:
    referencia = date.fromisoformat(config['data_referencia'])
except ValueError as erro:
    raise ValueError('Preencha data_referencia no formato AAAA-MM-DD (ex.: 2026-10-22)') from erro
if referencia.isoformat() != config['data_referencia']:
    raise ValueError('data_referencia deve usar o formato AAAA-MM-DD')
batch_id = config['batch_id'] or f'gold-{referencia:%Y%m%d}-{uuid.uuid4().hex[:8]}'
if not re.fullmatch(r'[A-Za-z0-9_.-]+', batch_id):
    raise ValueError(f'batch_id inválido: {batch_id}')
if config['modo'] == 'exportar':
    raiz_exportacao = Path(config['diretorio_exportacao'])
    if not str(raiz_exportacao).startswith('/Volumes/') or '..' in raiz_exportacao.parts:
        raise ValueError('Exportação exige diretorio_exportacao em /Volumes/catalogo/schema/volume')


def qualificar(nome, partes):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise ValueError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
raiz_contratos = Path(config['contratos']).resolve()
specs = ler_contratos(raiz_contratos)
gerar_manifesto = carregar_gerador_de_manifesto(raiz_contratos)
print(
    f'Schema Gold: {schema_gold} | Modo: {config["modo"]} | Regras: {VERSAO_REGRAS} | '
    f'Contrato: {VERSAO_CONTRATO} | Lote: {batch_id} | Referência: {referencia}'
)

Schema Gold: `v-energy`.`gold` | Modo: exportar | Regras: GOLD_EXPORTACAO_V1 | Contrato: 0.3.0-provisional | Lote: gold-20261022-8a899b6b | Referência: 2026-10-22


## 1. Leitura das 12 tabelas num retrato só

In [0]:
# ruff: noqa: F821
from gold_faturas import ler_publicacao_gold

fontes = {}


def ler_versao(tabela):
    """Uma versão Delta fixa: as 12 tabelas saem do mesmo retrato, mesmo com o Job rodando."""
    nome = f'{schema_gold}.`{tabela}`'
    if not spark.catalog.tableExists(nome):
        raise ValueError(f'Tabela Gold não publicada: {nome}')
    versao = spark.sql(f'DESCRIBE HISTORY {nome} LIMIT 1').first()['version']
    fontes[tabela] = {'tabela': nome, 'versao_delta': versao}
    print(f'{tabela}: {nome} (versão {versao})')
    return spark.read.option('versionAsOf', versao).table(nome)


tabelas = {nome: ler_versao(nome) for nome in specs if nome not in TABELAS_FATURAS}

# Gold de faturas: lida pelo marcador, com todas as revisões confirmadas na mesma execução
marcador = spark.table(f'{schema_gold}.`tb_gold_faturas_publicacao`').collect()
publicadas, linhas_marcador = ler_publicacao_gold(
    spark, config['schema_gold'], sorted({r['nome'] for r in marcador})
)
metadados_faturas = json.loads(linhas_marcador[0]['metadados'])
for nome in TABELAS_FATURAS:
    tabelas[nome] = publicadas[nome]
    linha = next(r for r in linhas_marcador if r['nome'] == nome)
    fontes[nome] = {'tabela': linha['fonte'], 'versao_delta': linha['versao_fonte']}
    print(f'{nome}: {linha["fonte"]} (versão {linha["versao_fonte"]}, pelo marcador)')

# Período do lote: a cobertura mensal de uc_consumo_faturamento_mensal, como cobra o importador
limites = (
    tabelas['uc_consumo_faturamento_mensal']
    .agg(F.min('competencia').alias('inicio'), F.max('competencia').alias('fim'))
    .first()
)
periodo = {'start': f'{limites["inicio"]:%Y-%m}', 'end': f'{limites["fim"]:%Y-%m}'}
print(f'Período do lote: {periodo["start"]} a {periodo["end"]}')

atendimentos: `v-energy`.`gold`.`atendimentos` (versão 1)
atendimentos_historico: `v-energy`.`gold`.`atendimentos_historico` (versão 1)
continuidade_alimentador_mensal: `v-energy`.`gold`.`continuidade_alimentador_mensal` (versão 0)
dim_alimentador: `v-energy`.`gold`.`dim_alimentador` (versão 0)
dim_uc: `v-energy`.`gold`.`dim_uc` (versão 1)
fila_inspecao: `v-energy`.`gold`.`fila_inspecao` (versão 3)
gd_uc: `v-energy`.`gold`.`gd_uc` (versão 3)
inspecoes_historicas: `v-energy`.`gold`.`inspecoes_historicas` (versão 3)
medidores_historico: `v-energy`.`gold`.`medidores_historico` (versão 0)
ocorrencias_uc: `v-energy`.`gold`.`ocorrencias_uc` (versão 0)
uc_consumo_faturamento_mensal: `v-energy`.`gold`.`tb_gold_faturas_uc_consumo_faturamento_mensal_revisao_bc27773a65af4bb68c118dd0f0858a16` (versão 0, pelo marcador)
receita_volume_mensal: `v-energy`.`gold`.`tb_gold_faturas_receita_volume_mensal_revisao_bc27773a65af4bb68c118dd0f0858a16` (versão 0, pelo marcador)
Período do lote: 2022-07 a 2026-06

## 2. Conferência do lote com as regras do importador

In [0]:
# ruff: noqa: F821
violacoes, linhas = {}, {}
for nome, spec in specs.items():
    df = tabelas[nome]
    divergencias = divergencias_de_esquema(df, spec)
    if divergencias:
        raise ValueError(f'{nome}: colunas, tipos ou ordem diferentes do YAML: {divergencias}')
    violacoes |= contar(df, regras_da_tabela(nome, spec))
    violacoes[f'{nome}: chave {", ".join(spec["primary_key"])} repetida'] = chave_repetida(df, spec)
    if 'competencia' in spec['columns']:
        fora = df.filter(
            (F.date_format('competencia', 'yyyy-MM') < periodo['start'])
            | (F.date_format('competencia', 'yyyy-MM') > periodo['end'])
        )
        violacoes[f'{nome}: competência fora do período do lote'] = fora.count()
    linhas[nome] = df.count()

violacoes |= referencias_orfas(tabelas, specs)
for nome, df in tabelas.items():
    df.createOrReplaceTempView(f'exp_{nome}')
for regra, consulta in PORTOES.items():
    violacoes[regra] = spark.sql(consulta).count()

# Mesma referência e mesmo período da Gold de faturas, que já exportou o recibo dela com eles
violacoes['lote: data_referencia diferente da Gold de faturas'] = int(
    metadados_faturas.get('reference_date') != config['data_referencia']
)
violacoes['lote: período diferente do da Gold de faturas'] = int(
    metadados_faturas.get('period') != periodo
)
# R6 do manifesto: a referência não pode ser anterior ao último mês do período
violacoes['lote: data_referencia antes do último mês do período (R6)'] = int(
    config['data_referencia'] < f'{periodo["end"]}-01'
)

display(
    spark.createDataFrame(
        [(regra, n, 'OK' if n == 0 else 'FALHA') for regra, n in violacoes.items()],
        'regra string, violacoes long, status string',
    )
)
for nome in specs:
    print(f'{nome}: {linhas[nome]:,} linhas')
falhas = {regra: n for regra, n in violacoes.items() if n}
if falhas:
    raise ValueError(f'Lote reprovado antes de exportar: {falhas}')
print(f'[OK] {len(violacoes)} regras, 12 tabelas conferidas para o lote {batch_id}')

regra,violacoes,status
atendimentos.protocolo: nulo,0,OK
atendimentos.protocolo: texto vazio,0,OK
atendimentos.id_uc: nulo,0,OK
atendimentos.id_uc: texto vazio,0,OK
atendimentos.id_uc: fora do padrão ^UC\d{6}$,0,OK
atendimentos.aberto_em: nulo,0,OK
atendimentos.aberto_em: fração de segundo,0,OK
atendimentos.assunto: nulo,0,OK
atendimentos.assunto: texto vazio,0,OK
atendimentos.status: nulo,0,OK


atendimentos: 318,871 linhas
atendimentos_historico: 636,857 linhas
continuidade_alimentador_mensal: 5,856 linhas
dim_alimentador: 122 linhas
dim_uc: 45,000 linhas
fila_inspecao: 141 linhas
gd_uc: 3,510 linhas
inspecoes_historicas: 1,389 linhas
medidores_historico: 51,299 linhas
ocorrencias_uc: 180,000 linhas
receita_volume_mensal: 63,930 linhas
uc_consumo_faturamento_mensal: 2,152,336 linhas
[OK] 223 regras, 12 tabelas conferidas para o lote gold-20261022-8a899b6b


## 3. Exportação: CSVs, proveniência e manifesto por último

In [0]:
# ruff: noqa: F821
if config['modo'] == 'validar':
    dbutils.notebook.exit(f'modo=validar: lote {batch_id} conferido, nada gravado')

pasta = raiz_exportacao / batch_id
pasta.mkdir(parents=True, exist_ok=False)
for nome, spec in specs.items():
    escritas = escrever_csv(tabelas[nome], spec, pasta / f'{nome}.csv')
    if escritas != linhas[nome]:
        raise ValueError(f'{nome}: {escritas:,} linhas no CSV, {linhas[nome]:,} na tabela')
    print(f'{nome}.csv: {escritas:,} linhas')

# Proveniência antes do manifesto: versões Delta lidas e a execução da Gold de faturas
(pasta / 'fontes.json').write_text(
    json.dumps(
        {
            'batch_id': batch_id,
            'regras': VERSAO_REGRAS,
            'fontes': fontes,
            'faturas_id_execucao': linhas_marcador[0]['id_execucao'],
        },
        ensure_ascii=False,
        indent=2,
    )
    + '\n'
)

# O manifest.json vai por último: sem ele, o importador não aceita a pasta
manifesto = gerar_manifesto(
    pasta,
    batch_id=batch_id,
    periodo=f'{periodo["start"]}..{periodo["end"]}',
    data_referencia=config['data_referencia'],
    sintetico=False,
)
for nome, spec in specs.items():
    entrada = manifesto['tables'][nome]
    if entrada['rows'] != linhas[nome]:
        raise ValueError(
            f'{nome}: manifesto com {entrada["rows"]:,} linhas, tabela com {linhas[nome]:,}'
        )
    esperado = totais_de_controle(tabelas[nome], spec)
    if {c: Decimal(v) for c, v in entrada['totals'].items()} != esperado:
        raise ValueError(f'{nome}: totais do CSV diferentes da tabela: {entrada["totals"]}')
print(f'Lote exportado em {pasta}, com manifest.json e fontes.json')
print('Antes de carregar no portal: importador --origem <pasta> --verificar')

atendimentos.csv: 318,871 linhas
atendimentos_historico.csv: 636,857 linhas
continuidade_alimentador_mensal.csv: 5,856 linhas
dim_alimentador.csv: 122 linhas
dim_uc.csv: 45,000 linhas
fila_inspecao.csv: 141 linhas
gd_uc.csv: 3,510 linhas
inspecoes_historicas.csv: 1,389 linhas
medidores_historico.csv: 51,299 linhas
ocorrencias_uc.csv: 180,000 linhas
receita_volume_mensal.csv: 63,930 linhas
uc_consumo_faturamento_mensal.csv: 2,152,336 linhas
Lote exportado em /Volumes/v-energy/gold/exportacao/gold-20261022-8a899b6b, com manifest.json e fontes.json
Antes de carregar no portal: importador --origem <pasta> --verificar
